# Notebook used to test the models

### Packages Import


In [2]:
%load_ext autoreload
%autoreload 2

from ml_powerlifting_potential.data import load_dataset
from sklearn.model_selection import train_test_split
from ml_powerlifting_potential.config import FEATURES, TARGETS, SEED
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
import time

### Data Import & Train/Test Split

In [9]:
dataset = load_dataset()
dataset.describe()  # min, max, moyenne, quartiles de chaque colonne
dataset.isna().sum()  # doit afficher 0 partout

X = dataset[FEATURES]
y_squat = dataset[TARGETS[0]]
y_deadlift = dataset[TARGETS[1]]
X_train, X_test, y_squat_train, y_squat_test, y_deadlift_train, y_deadlift_test = (
    train_test_split(X, y_squat, y_deadlift, test_size=0.2, random_state=SEED)
)

bench_train = X_train["Best3BenchKg"]
bench_test = X_test["Best3BenchKg"]

### Error Evaluation Method

In [10]:
def evaluate(y_true, y_pred) -> dict[str, float]:
    """MAE in kg (average error) and R2 (1 = perfect, 0 = no better than the mean)."""
    return {"MAE": mean_absolute_error(y_true, y_pred), "R2": r2_score(y_true, y_pred)}

### Ratio Estimator Method

In [ ]:
# Gym rule: median of the per-athlete ratios, computed on the TRAIN set only
ratio_squat = (y_squat_train / bench_train).median()
ratio_deadlift = (y_deadlift_train / bench_train).median()
print(f"squat/bench = {ratio_squat:.2f}, deadlift/bench = {ratio_deadlift:.2f}")

# Predictions on the TEST set
pred_squat_rule = bench_test * ratio_squat
pred_deadlift_rule = bench_test * ratio_deadlift


print("Squat rule:", evaluate(y_squat_test, pred_squat_rule))
print("Deadlift rule:", evaluate(y_deadlift_test, pred_deadlift_rule))

squat/bench = 1.57, deadlift/bench = 1.85
Squat rule: {'MAE': 21.590190954021267, 'R2': 0.7480737990839819}
Deadlift rule: {'MAE': 28.404329357495882, 'R2': 0.5838059452247737}


### Linear Regression Model

In [17]:
linear_model_squat = LinearRegression()
linear_model_squat.fit(X_train, y_squat_train)

linear_model_deadlift = LinearRegression()
linear_model_deadlift.fit(X_train, y_deadlift_train)

pred_squat_linear = linear_model_squat.predict(X_test)
pred_deadlift_linear = linear_model_deadlift.predict(X_test)

print("Squat linear:", evaluate(y_squat_test, pred_squat_linear))
print("Deadlift linear:", evaluate(y_deadlift_test, pred_deadlift_linear))

Squat linear: {'MAE': 17.624369244378496, 'R2': 0.8310389597243819}
Deadlift linear: {'MAE': 19.318098649301696, 'R2': 0.8073293069867008}


### Polynomial Regression Model

In [16]:
def make_poly_pipeline(degree: int) -> Pipeline:
    """Polynomial features -> standardization -> linear regression."""
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
            ("scale", StandardScaler()),
            ("linreg", LinearRegression()),
        ]
    )


targets = {
    "squat": (y_squat_train, y_squat_test),
    "deadlift": (y_deadlift_train, y_deadlift_test),
}

results = []
for degree in [2, 3]:
    for target_name, (y_train, y_test) in targets.items():
        model = make_poly_pipeline(degree)
        model.fit(X_train, y_train)
        for split_name, X_part, y_part in [
            ("train", X_train, y_train),
            ("test", X_test, y_test),
        ]:
            scores = evaluate(y_part, model.predict(X_part))
            results.append(
                {
                    "model": f"poly{degree}",
                    "target": target_name,
                    "split": split_name,
                    **scores,
                }
            )

results_df = pd.DataFrame(results)
results_df

,model,target,split,MAE,R2
0,poly2,squat,train,17.329804,0.835871
1,poly2,squat,test,17.343826,0.836093
2,poly2,deadlift,train,18.673604,0.819935
3,poly2,deadlift,test,18.720340,0.818673
4,poly3,squat,train,17.279228,0.836732
5,poly3,squat,test,17.315425,0.836684
6,poly3,deadlift,train,18.578508,0.821771
7,poly3,deadlift,test,18.640255,0.820338


### Support Vector Machine

In [4]:
def make_svr(
    C: float = 10, gamma: float = 0.2, epsilon: float = 0.1
) -> TransformedTargetRegressor:
    """RBF SVR with standardized inputs AND standardized target."""
    return TransformedTargetRegressor(
        regressor=Pipeline(
            [
                ("scale", StandardScaler()),
                ("svr", SVR(kernel="rbf", C=C, gamma=gamma, epsilon=epsilon)),
            ]
        ),
        transformer=StandardScaler(),  # target scaled for training, converted back to kg for predictions
    )

### Finding the best SVR params using GridSearch

In [13]:
# values to try for each setting
param_grid = {
    "regressor__svr__C": [1, 10, 100],
    "regressor__svr__gamma": [0.05, 0.2, 0.5],
    "regressor__svr__epsilon": [0.05, 0.1],
}

# try every combination with 3-fold cross validation
grid_search = GridSearchCV(
    make_svr(),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",  # negative because sklearn looks for the highest score
    n_jobs=-1,  # use all cpu cores
)

# tune on 8000 lifters only, the full train set would take too long
grid_search.fit(X_train.iloc[:8_000], y_squat_train.iloc[:8_000])
best_SVR_C, best_SVR_epsilon, best_SVR_gamma = grid_search.best_params_.values()
print(grid_search.best_params_)
print(-grid_search.best_score_)  # best MAE in kg

{'regressor__svr__C': 10, 'regressor__svr__epsilon': 0.1, 'regressor__svr__gamma': 0.05}
17.25359349726295


In [14]:
# second pass, grid moved toward smaller gamma and bigger epsilon
param_grid = {
    "regressor__svr__C": [5, 10, 30],
    "regressor__svr__gamma": [0.01, 0.02, 0.05],
    "regressor__svr__epsilon": [0.1, 0.2],
}

grid_search = GridSearchCV(
    make_svr(),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_search.fit(X_train.iloc[:8_000], y_squat_train.iloc[:8_000])

best_SVR_C, best_SVR_epsilon, best_SVR_gamma = grid_search.best_params_.values()

print(grid_search.best_params_)
print(-grid_search.best_score_)

{'regressor__svr__C': 30, 'regressor__svr__epsilon': 0.1, 'regressor__svr__gamma': 0.02}
17.22780060294379


In [19]:
# best svr from the grid search, trained on 20000 lifters
best_svr = make_svr(C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon)
best_svr.fit(X_train.iloc[:20_000], y_squat_train.iloc[:20_000])

# evaluate on the full test set
print(evaluate(y_squat_test, best_svr.predict(X_test)))

{'MAE': 17.30984299163326, 'R2': 0.8363328725008987}


### 

### Evaluation of different processing time 

In [18]:
data_sizes = [2_000, 5_000, 10_000, 20_000]
X_eval, y_eval = X_test.iloc[:10_000], y_squat_test.iloc[:10_000]

timings = []
for n in data_sizes:
    X_sub, y_sub = (
        X_train.iloc[:n],
        y_squat_train.iloc[:n],
    )  # first n athletes (the split already shuffled them)
    for name, model in [
        ("linear", LinearRegression()),
        ("poly2", make_poly_pipeline(degree=2)),
        ("svr", make_svr(C=best_SVR_C, gamma=best_SVR_gamma, epsilon=best_SVR_epsilon)),
    ]:
        start = time.perf_counter()  # start the chronometer
        model.fit(X_sub, y_sub)
        fit_seconds = time.perf_counter() - start  # elapsed seconds
        mae = evaluate(y_eval, model.predict(X_eval))["MAE"]
        timings.append(
            {"model": name, "n_train": n, "fit_seconds": fit_seconds, "MAE": mae}
        )

timings_df = pd.DataFrame(timings)
timings_df

,model,n_train,fit_seconds,MAE
0,linear,2000,0.002275,17.464526
1,poly2,2000,0.010186,17.145360
2,svr,2000,0.206325,17.219544
3,linear,5000,0.003994,17.444398
4,poly2,5000,0.009024,17.131119
5,svr,5000,1.599353,17.140672
6,linear,10000,0.009064,17.425873
7,poly2,10000,0.035853,17.117821
8,svr,10000,10.579509,17.091743
9,linear,20000,0.011050,17.424997
